# Step 4. 산불발생 공간·지형·접근성 최종 EDA

이 노트북은 `jsw/강원_재_EDA` 폴더의 마지막 EDA 단계다.

Step 1~3에서 기상, 캐나다 산불지수, 매칭 대조군, 선행 기상, 국지 임계치 후보를 이미 정리했으므로 Step 4에서는 공간·지형·토지피복·접근성과 공간 대조군만 확인한다.

기존 Step 5와 Step 6은 이 EDA 흐름에서 제거한다. 도로·임도·등산로·생활권·산림 내부 같은 인간활동 프록시 요소는 별도 원인 분석이 아니라 Step 4의 접근성·공간층 변수로 흡수한다.

결과 해석은 이 노트북에 작성하지 않는다. 표와 플롯을 함께 검토한 해석, 한계, 다음 반영사항은 `Step4_산불발생_공간지형및대조군_분석_진행예정로그.md`에만 기록한다.

In [1]:
from pathlib import Path
import runpy
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
OUT_DIR = MODULE_DIR / "outputs/Step4"
TABLE_DIR = OUT_DIR / "tables"
PLOT_DIR = OUT_DIR / "plots"

## S4-01. 공간 원천·CRS·geometry 품질 감사

공간 원천 파일 존재 여부, CRS, bounds, geometry validity, 대용량 GPKG 메타데이터와 표본 geometry 유효성을 감사한다.

In [2]:
runpy.run_path(str(MODULE_DIR / "step4_s401_spatial_audit.py"), run_name="__main__")

{'source_rows': 12, 'geometry_rows': 13, 'non_convertible_layers': 0, 'invalid_full_geometry_count': 0, 'plot': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\outputs\\Step4\\plots\\S4-01_layer_overlay_quality_map.png'}


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s401_spatial_audit.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, loca

### S4-01 결과표 확인

In [3]:
for name in [
    "S4-01_spatial_source_audit.csv",
    "S4-01_geometry_validity_audit.csv",
    "S4-01_layer_bounds_audit.csv",
]:
    print("\n---", name)
    display(pd.read_csv(TABLE_DIR / name, encoding="utf-8-sig"))


--- S4-01_spatial_source_audit.csv


,key,label,kind,path,exists,bytes,modified,note
0,fire,raw_fire_events,csv_point_wgs84,data\강원도_데이터\강원도_산불발생.csv,True,408260,2026-05-31T04:59:51.900885105,NaN
1,clean_fire,clean_fire_events,csv_point_wgs84,data\학습데이터\산불발생_정제.csv,True,338762,2026-06-11T13:05:44.715772152,reference only; Step 4 spatial audit uses raw ...
2,weather_grid,weather_cell_polygons,csv_wkt_polygon_wgs84,data\강원도_날씨데이터\강원도날씨_격자.csv,True,48182,2026-05-30T10:28:16.485248327,NaN
3,climate_type,climate_topography_type,csv_table,data\강원도_날씨데이터\강원도날씨_기후지형유형_셀분류.csv,True,3111,2026-05-31T10:34:54.482622385,NaN
4,terrain,fire_terrain_features,csv_point_wgs84,data\강원도_데이터\산불_공간데이터\강원도_산불_지형특성계산.csv,True,477820,2026-05-31T05:05:06.908538103,NaN
5,dem,gangwon_dem,raster,data\강원도_데이터\강원도_공간데이터\강원도_DEM_데이터.tif,True,9933156,2026-05-31T05:21:43.481648207,NaN
6,landcover,landcover_fine_gpkg,gpkg,data\강원도_데이터\강원도_공간데이터\강원도_토지피복도_세분류_병합_1m.gpkg,True,1230585856,2026-05-31T07:09:51.222411394,large file; full metadata plus sample geometry...
7,roads,merged_roads_gpkg,gpkg,data\강원도_데이터\강원도_공간데이터\강원도_병합_도로.gpkg,True,307200000,2026-05-31T07:43:12.656644583,large file; full metadata plus sample geometry...
8,trails,hiking_trails,csv_wkt_line_wgs84,data\강원도_데이터\강원도_공간데이터\강원도_등산로.csv,True,20976258,2026-05-31T05:06:41.769750834,NaN
9,forest_roads,forest_roads,csv_wkt_line_wgs84,data\강원도_데이터\강원도_공간데이터\강원도_임도망도.csv,True,20156788,2026-05-31T05:13:49.485568762,NaN



--- S4-01_geometry_validity_audit.csv


,key,label,audit_scope,feature_count,crs,target_crs_convertible,geometry_types,null_geometry_count,empty_geometry_count,invalid_geometry_count,minx,miny,maxx,maxy,width,height,resolution_x,resolution_y,nodata
0,fire,raw_fire_events,full,3405.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.144220,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
1,clean_fire,clean_fire_events,full,1558.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.160133,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
2,weather_grid,weather_cell_polygons,full,92.0,EPSG:4326,True,"MultiPolygon,Polygon",0.0,0.0,0.0,127.095000,3.702780e+01,1.293657e+02,3.861180e+01,NaN,NaN,NaN,NaN,NaN
3,terrain,fire_terrain_features,full,3405.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.144220,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
4,dem,gangwon_dem,raster_metadata,NaN,"PROJCS[""Transverse Mercator"",GEOGCS[""GRS80 ELL...",True,Raster,NaN,NaN,NaN,208276.000000,4.935230e+05,4.101460e+05,6.688430e+05,2243.0,1948.0,90.0,90.0,-9999.0
5,landcover,landcover_fine_gpkg,full_metadata,41119.0,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,MultiPolygon,NaN,NaN,NaN,217524.670000,4.930995e+05,4.110727e+05,6.427261e+05,NaN,NaN,NaN,NaN,NaN
6,landcover,landcover_fine_gpkg,first_5000_sample,5000.0,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,MultiPolygon,0.0,0.0,0.0,241720.540000,5.530424e+05,2.882790e+05,6.004719e+05,NaN,NaN,NaN,NaN,NaN
7,roads,merged_roads_gpkg,full_metadata,91862.0,EPSG:5179,True,Unknown,NaN,NaN,NaN,965866.352115,1.892516e+06,1.166322e+06,2.069685e+06,NaN,NaN,NaN,NaN,NaN
8,roads,merged_roads_gpkg,first_5000_sample,5000.0,EPSG:5179,True,"MultiPolygon,Polygon",0.0,0.0,0.0,973716.312804,1.892516e+06,1.166322e+06,2.042086e+06,NaN,NaN,NaN,NaN,NaN
9,trails,hiking_trails,full,4523.0,EPSG:4326,True,"LineString,MultiLineString",0.0,0.0,0.0,127.175942,3.702799e+01,1.293549e+02,3.852591e+01,NaN,NaN,NaN,NaN,NaN



--- S4-01_layer_bounds_audit.csv


,key,label,audit_scope,crs,target_crs_convertible,minx,miny,maxx,maxy
0,fire,raw_fire_events,full,EPSG:4326,True,127.144220,3.706870e+01,1.293513e+02,3.858611e+01
1,clean_fire,clean_fire_events,full,EPSG:4326,True,127.160133,3.706870e+01,1.293513e+02,3.858611e+01
2,weather_grid,weather_cell_polygons,full,EPSG:4326,True,127.095000,3.702780e+01,1.293657e+02,3.861180e+01
3,terrain,fire_terrain_features,full,EPSG:4326,True,127.144220,3.706870e+01,1.293513e+02,3.858611e+01
4,dem,gangwon_dem,raster_metadata,"PROJCS[""Transverse Mercator"",GEOGCS[""GRS80 ELL...",True,208276.000000,4.935230e+05,4.101460e+05,6.688430e+05
5,landcover,landcover_fine_gpkg,full_metadata,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,217524.670000,4.930995e+05,4.110727e+05,6.427261e+05
6,landcover,landcover_fine_gpkg,first_5000_sample,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,241720.540000,5.530424e+05,2.882790e+05,6.004719e+05
7,roads,merged_roads_gpkg,full_metadata,EPSG:5179,True,965866.352115,1.892516e+06,1.166322e+06,2.069685e+06
8,roads,merged_roads_gpkg,first_5000_sample,EPSG:5179,True,973716.312804,1.892516e+06,1.166322e+06,2.042086e+06
9,trails,hiking_trails,full,EPSG:4326,True,127.175942,3.702799e+01,1.293549e+02,3.852591e+01


## S4-02 : 발생지 토지피복·지형 변수 결합

산불 발생지에 토지피복 분류와 지형 특성 데이터를 결합하고, 감사 및 기술통계를 생성합니다.

In [4]:
runpy.run_path(str(MODULE_DIR / "step4_s402_landcover_terrain.py"), run_name="__main__")

--- S4-02: 발생지 토지피복·지형 변수 결합 시작 ---
데이터 로딩 중...


정제 산불 행 수: 1558
지형특성 행 수: 3405
토지피복 폴리곤 수: 41119
지형특성 결합 후 행 수: 1558


토지피복도 공간조인 실행 중...


최종 처리 후 행 수: 1558

--- 공간조인 감사 결과 ---
                   metric  value   ratio_pct
0         모집단 행 수 (정제 산불)   1558  100.000000
1        공간조인 전 임시 매칭 행 수   1569  100.706033
2       다중 매칭(중복) 발생 사건 수     11    0.706033
3  토지피복 공간조인 실패(미매칭) 사건 수    163   10.462131
4   토지피복 공간조인 성공(매칭) 사건 수   1395   89.537869
5             최종 처리 후 행 수   1558  100.000000

--- 번지유형과 실제 토지피복 교차표 ---
           산림 아님(n)   산림 아님(%)  산림지역(n)    산림지역(%)  Total(n)
addr_type                                                   
일반번지           1054  94.275492       64   5.724508      1118
임야번지(산)         246  55.909091      194  44.090909       440
Total          1300  83.440308      258  16.559692      1558

--- 지형 변수 결측 감사 ---
고도(m)            0
경사도(도)           0
TPI(지형위치지수)      0
TWI(지형다습지수)    115
dtype: int64

--- 기후지형유형별 지형 변수 요약표 ---
       variable  기후지형유형  count          mean           std        min  \
0         고도(m)  고지·산간형  223.0  5.331049e+02  1.794650e+02  93.284540   
1         고도(m)  영동 해안형  563.0  

S4-02_landcover_composition.png 생성 완료


S4-02_terrain_ecdf_by_type_elevation.png 생성 완료


S4-02_terrain_ecdf_by_type_slope.png 생성 완료


S4-02_terrain_ecdf_by_type_tpi.png 생성 완료
S4-02_terrain_ecdf_by_type_twi.png 생성 완료
--- S4-02: 발생지 토지피복·지형 변수 결합 완료 ---


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s402_landcover_terrain.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, 

### S4-02 결과 확인

In [5]:
print("\n--- 공간조인 감사 결과 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_fire_landcover_join_audit.csv", encoding="utf-8-sig"))

print("\n--- 번지유형과 실제 토지피복 교차표 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_address_forest_landcover_crosstab.csv", index_col=0, encoding="utf-8-sig"))

print("\n--- 기후지형유형별 지형 변수 요약표 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_fire_terrain_summary.csv", encoding="utf-8-sig"))


--- 공간조인 감사 결과 ---


,metric,value,ratio_pct
0,모집단 행 수 (정제 산불),1558,100.000000
1,공간조인 전 임시 매칭 행 수,1569,100.706033
2,다중 매칭(중복) 발생 사건 수,11,0.706033
3,토지피복 공간조인 실패(미매칭) 사건 수,163,10.462131
4,토지피복 공간조인 성공(매칭) 사건 수,1395,89.537869
5,최종 처리 후 행 수,1558,100.000000



--- 번지유형과 실제 토지피복 교차표 ---


,산림 아님(n),산림 아님(%),산림지역(n),산림지역(%),Total(n)
addr_type,,,,,
일반번지,1054,94.275492,64,5.724508,1118
임야번지(산),246,55.909091,194,44.090909,440
Total,1300,83.440308,258,16.559692,1558



--- 기후지형유형별 지형 변수 요약표 ---


,variable,기후지형유형,count,mean,std,min,25%,50%,75%,max
0,고도(m),고지·산간형,223.0,5.331049e+02,1.794650e+02,93.284540,418.843290,550.386350,658.588020,9.891533e+02
1,고도(m),영동 해안형,563.0,7.004671e+01,8.460261e+01,1.306317,15.526400,44.794773,86.489360,5.426060e+02
2,고도(m),영서 내륙형,772.0,2.820546e+02,1.575908e+02,52.604774,167.229502,239.888285,370.246090,1.163781e+03
3,경사도(도),고지·산간형,223.0,9.115672e+00,6.440241e+00,0.111244,3.869198,8.034257,12.927752,2.992135e+01
4,경사도(도),영동 해안형,563.0,4.739330e+00,4.114957e+00,0.053691,1.731844,3.939325,6.293261,3.236751e+01
5,경사도(도),영서 내륙형,772.0,7.491882e+00,5.670407e+00,0.080065,3.108447,6.246531,10.495640,2.996693e+01
6,TPI(지형위치지수),고지·산간형,223.0,-3.474520e+00,5.736750e+00,-19.685455,-7.200684,-2.938171,-0.365997,1.832538e+01
7,TPI(지형위치지수),영동 해안형,563.0,-1.551318e+00,3.896519e+00,-15.957672,-2.994246,-0.996704,0.078687,1.619081e+01
8,TPI(지형위치지수),영서 내륙형,772.0,-1.772430e+00,5.197813e+00,-27.303452,-3.903320,-1.575827,-0.080238,3.567560e+01
9,TWI(지형다습지수),고지·산간형,219.0,1.250841e+08,4.321624e+08,1.360585,3.522593,4.711554,6.510787,1.611377e+09


## S4-03 : 접근성·공간층 변수 생성

주요 인프라 최단거리 변수의 정합성을 검증하고, 이를 토대로 WUI/산림 접근권/산림 내부 공간층을 할당하며 민감도 분석을 수행합니다.

In [6]:
runpy.run_path(str(MODULE_DIR / "step4_s403_accessibility.py"), run_name="__main__")

--- S4-03: 접근성·공간층 변수 생성 시작 ---
로드된 산불 공간 피처 행 수: 1558
도로 및 등산로/임도 최단거리 정합성 샘플 검증 진행 중...


샘플 20건 최단거리 교차 검증 결과:
  [F_003556] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007615] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007616] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007617] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007618] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007619] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007620] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007621] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007622] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007623] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007624] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007625] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007626] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007627] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007628] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007629] 도로 오차: 0.0000m | 임도 

S4-03_accessibility_ecdf.png 생성 완료


S4-03_spatial_layer_map.png 생성 완료
--- S4-03: 접근성·공간층 변수 생성 완료 ---


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s403_accessibility.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, loca

### S4-03 결과 확인

In [7]:
print("\n--- 임계치별 공간층 구성비 민감도 분석표 ---")
display(pd.read_csv(TABLE_DIR / "S4-03_access_threshold_sensitivity.csv", encoding="utf-8-sig"))

print("\n--- 기후지형유형별 접근성 변수 요약표 ---")
display(pd.read_csv(TABLE_DIR / "S4-03_accessibility_distance_summary.csv", encoding="utf-8-sig"))


--- 임계치별 공간층 구성비 민감도 분석표 ---


,임계치_m,생활권-WUI(n),생활권-WUI(%),산림 접근권(n),산림 접근권(%),산림 내부(n),산림 내부(%),Total(n)
0,250,1464,93.966624,26,1.668806,68,4.364570,1558
1,500,1464,93.966624,31,1.989730,63,4.043646,1558
2,1000,1464,93.966624,38,2.439024,56,3.594352,1558



--- 기후지형유형별 접근성 변수 요약표 ---


,variable,기후지형유형,count,mean,std,min,25%,50%,75%,max
0,도로_최단거리_m,고지·산간형,223.0,30.428069,146.576288,0.000000,0.000000,2.290541,17.255557,1840.265315
1,도로_최단거리_m,영동 해안형,563.0,21.187855,72.742159,0.000000,0.000000,3.079896,11.698024,936.566738
2,도로_최단거리_m,영서 내륙형,772.0,61.238639,297.718639,0.000000,0.000000,4.610877,19.072675,4774.742876
3,임도_최단거리_m,고지·산간형,223.0,2377.932024,3140.975814,0.424022,921.017376,1765.700719,2910.438781,27048.663273
4,임도_최단거리_m,영동 해안형,563.0,5056.299855,2936.277785,0.796963,3028.163086,4570.900509,6550.005849,26968.592033
5,임도_최단거리_m,영서 내륙형,772.0,4780.638426,3972.977848,1.132785,1801.281840,3525.646592,6694.701049,18951.273258
6,등산로_최단거리_m,고지·산간형,223.0,1390.346699,1572.273826,1.258764,334.872619,855.056345,1912.874537,8793.677915
7,등산로_최단거리_m,영동 해안형,563.0,1086.507637,958.503604,0.619048,332.624061,825.751613,1560.911314,7188.978270
8,등산로_최단거리_m,영서 내륙형,772.0,2527.003633,2346.781788,0.073597,737.630865,1945.442042,3538.180668,16883.667164
9,시가화_최단거리_m,고지·산간형,223.0,371.656635,2841.410401,0.000000,0.000000,1.615978,14.082315,25759.931131


## 다음 구현 범위

1. S4-04에서 동일 기상셀 또는 층화 공간 대조군 후보풀을 생성하고 편향 여부를 확인한다.
2. S4-05에서 발생지 대 공간 대조군 최소 비교를 통해 변수별 효과크기와 편향을 검정한다.
3. Step 4 결과를 최종 EDA 요약으로 정리한다.